# 🖼️ Computer Vision with Deep Learning: CNN & Transfer Learning

## Overview
This notebook demonstrates three Computer Vision (CV) approaches for image classification on **CIFAR-10**:

1. **CNN from Scratch** — Build and train a custom convolutional neural network
2. **VGG16 Frozen** — Use ImageNet pretrained weights with a frozen backbone
3. **VGG16 Fine-tuned** — Unfreeze the top layers for task-specific adaptation

**Dataset:** CIFAR-10 (60,000 × 32×32 color images, 10 classes)

## 1️⃣ Import Libraries

In [1]:
# Core libraries for deep learning and visualization
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow version:", tf.__version__)

# CIFAR-10 class labels
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

TensorFlow version: 2.15.0


## 2️⃣ Load & Explore Dataset (CIFAR-10)

CIFAR-10 consists of **60,000 color images** (32×32 pixels) in **10 classes**, with 6,000 images per class.  
The dataset is split into 50,000 training and 10,000 test images.

In [2]:
# Load CIFAR-10 dataset (60,000 32x32 color images in 10 classes)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.ravel()
y_test = y_test.ravel()

print(f"Training set:   {x_train.shape}, labels: {y_train.shape}")
print(f"Test set:       {x_test.shape}, labels: {y_test.shape}")
print(f"Number of classes: {len(CLASS_NAMES)}")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Training set:   (50000, 32, 32, 3), labels: (50000,)
Test set:       (10000, 32, 32, 3), labels: (10000,)
Number of classes: 10


In [ ]:
# Visualize one sample image per class (2 rows x 5 cols)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.ravel()

for class_id in range(10):
    idx = np.where(y_train == class_id)[0][0]
    axes[class_id].imshow(x_train[idx])
    axes[class_id].set_title(CLASS_NAMES[class_id], fontsize=11)
    axes[class_id].axis('off')

plt.suptitle('CIFAR-10 Sample Images — One per Class', fontsize=14)
plt.tight_layout()
plt.show()

## 3️⃣ Preprocess Data

In [3]:
# Resize images to 224x224 for VGG16 compatibility and normalize to [0, 1]
IMG_SIZE = (224, 224)
BATCH = 32  # Reduced from 64 for lower GPU memory usage; increase if resources allow

def preprocess(x, y, augment=False):
    """Resize, normalize, and optionally augment images."""
    x = tf.image.resize(x, IMG_SIZE)
    x = tf.cast(x, tf.float32) / 255.0
    if augment:
        x = tf.image.random_flip_left_right(x)
        x = tf.image.random_brightness(x, 0.1)
    return x, y

# Build tf.data pipelines
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(20000)
    .map(lambda x, y: preprocess(x, y, augment=True))
    .batch(BATCH).prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(lambda x, y: preprocess(x, y, augment=False))
    .batch(BATCH).prefetch(tf.data.AUTOTUNE)
)

print("Data pipelines ready.")

Data pipelines ready.


## 4️⃣ Model 1: CNN from Scratch

We build a simple CNN architecture with 3 convolutional blocks followed by a classification head.  
**Expected accuracy:** ~60% after 3 epochs.

In [ ]:
def build_scratch():
    """Build a simple CNN from scratch for CIFAR-10 classification."""
    inputs = layers.Input(shape=(224, 224, 3))
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = layers.GlobalAveragePooling2D()(x)  # Better than Flatten: reduces params, reduces overfitting
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    return models.Model(inputs, outputs, name='CNN_Scratch')

model_scratch = build_scratch()
model_scratch.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model_scratch.summary()

In [ ]:
# Train CNN from scratch
print("Training CNN from scratch...")
history_scratch = model_scratch.fit(
    train_ds,
    epochs=3,
    validation_data=test_ds,
    verbose=1
)

scratch_acc = history_scratch.history['val_accuracy'][-1]
print(f"\n✅ CNN Scratch — Final Validation Accuracy: {scratch_acc:.4f}")

## 5️⃣ Model 2: VGG16 with Frozen Weights (Transfer Learning)

**Transfer learning** leverages knowledge learned on a large dataset (ImageNet, 1.2M images) and applies it to our smaller dataset.  
By **freezing** the backbone, we only train the new classification head.  
**Expected accuracy:** ~50–65% (limited adaptation).

In [ ]:
# Load VGG16 pretrained on ImageNet (without top classification layers)
base_vgg = tf.keras.applications.VGG16(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3),
    pooling='avg'
)

# Freeze all base model layers
base_vgg.trainable = False

# Build transfer learning model
inputs = layers.Input(shape=(224, 224, 3))
x = base_vgg(inputs, training=False)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation='softmax')(x)
model_frozen = models.Model(inputs, outputs, name='VGG16_Frozen')

model_frozen.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model_frozen.summary()

In [ ]:
# Train VGG16 with frozen base
print("Training VGG16 with frozen weights...")
history_frozen = model_frozen.fit(
    train_ds,
    epochs=3,
    validation_data=test_ds,
    verbose=1
)

frozen_acc = history_frozen.history['val_accuracy'][-1]
print(f"\n✅ VGG16 Frozen — Final Validation Accuracy: {frozen_acc:.4f}")

## 6️⃣ Model 3: VGG16 Fine-Tuned

**Fine-tuning** unfreezes the top portion of the pretrained backbone, allowing those layers to adapt to CIFAR-10.  
We use a **very small learning rate** (1e-5) to avoid destroying the pretrained weights.  
**Expected accuracy:** ~82% after fine-tuning.

In [ ]:
# Unfreeze the last 30% of VGG16 layers for fine-tuning
base_vgg.trainable = True
fine_tune_at = int(len(base_vgg.layers) * 0.7)

for i, layer in enumerate(base_vgg.layers):
    layer.trainable = (i >= fine_tune_at)

# Recompile with a much smaller learning rate
model_frozen.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Fine-tuning from layer {fine_tune_at} of {len(base_vgg.layers)}")
print("Training fine-tuned VGG16...")

history_finetune = model_frozen.fit(
    train_ds,
    epochs=5,
    validation_data=test_ds,
    verbose=1
)

finetune_acc = history_finetune.history['val_accuracy'][-1]
print(f"\n✅ VGG16 Fine-tuned — Final Validation Accuracy: {finetune_acc:.4f}")

## 7️⃣ Confusion Matrix for Best Model

The confusion matrix shows which classes the model confuses most often, helping identify areas for further improvement.

In [ ]:
# Generate predictions on the test set using the fine-tuned model
print("Generating predictions for confusion matrix...")
y_pred_probs = model_frozen.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.title('Confusion Matrix — VGG16 Fine-tuned (Best Model)', fontsize=13)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

## 8️⃣ Model Comparison Summary


In [ ]:
# Model performance comparison table
comparison_data = {
    'Model': ['CNN from Scratch', 'VGG16 Frozen', 'VGG16 Fine-tuned'],
    'Approach': ['Train from scratch', 'Transfer Learning (frozen)', 'Transfer Learning (fine-tuned)'],
    'Val Accuracy (%)': [
        round(scratch_acc * 100, 2),
        round(frozen_acc * 100, 2),
        round(finetune_acc * 100, 2)
    ],
    'Training Epochs': [3, 3, 5]
}

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

best_idx = comparison_df['Val Accuracy (%)'].idxmax()
print(f"\n🏆 Best Model: {comparison_df.loc[best_idx, 'Model']} "
      f"({comparison_df.loc[best_idx, 'Val Accuracy (%)']}%)")

## 🔑 Key Takeaways

| Concept | Insight |
|---|---|
| **Training from scratch** | Achieves ~60% with limited data and short training |
| **Frozen transfer learning** | May underperform without fine-tuning due to domain gap |
| **Fine-tuning** | Significantly boosts performance by adapting to the target domain |
| **Data augmentation** | Random flips and brightness changes improve generalization |
| **Learning rate** | Use 1e-5 or lower when fine-tuning to preserve pretrained features |

### Conclusion
Fine-tuned VGG16 dramatically outperforms training from scratch, demonstrating the power of transfer learning.  
For production image classification tasks with limited data, fine-tuning a pretrained backbone is the recommended approach.